# TriML - Full Pipeline
**CS 6140 Machine Learning | Northeastern**

Run everything here on Colab with a T4 GPU. Set runtime to GPU before running.

In [ ]:
!pip install -q torch scikit-learn pandas numpy matplotlib seaborn scipy

## Download data from Zenodo

In [ ]:
import urllib.request
import os

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

urls = {
    "athletes.csv": "https://zenodo.org/api/records/15401061/files/athletes.csv/content",
    "daily_data.csv": "https://zenodo.org/api/records/15401061/files/daily_data.csv/content",
    "activity_data.csv": "https://zenodo.org/api/records/15401061/files/activity_data.csv/content",
}

for fname, url in urls.items():
    dest = os.path.join(DATA_DIR, fname)
    if os.path.exists(dest):
        print(f"{fname} already exists")
        continue
    print(f"Downloading {fname}...", end=" ", flush=True)
    urllib.request.urlretrieve(url, dest)
    print(f"done ({os.path.getsize(dest)/1e6:.1f} MB)")

!ls -lh /content/data/

## Load and parse CSVs

In [ ]:
import ast, re, time, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, mean_absolute_error, mean_squared_error,
    precision_score, r2_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# parsers for the weird string columns in the CSVs

def _parse_hrv_range(s):
    nums = re.findall(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", str(s))
    return float(nums[0]), float(nums[1])

def _parse_hr_zones_athlete(s):
    cleaned = re.sub(r"np\.float64\(([^)]+)\)", r"\1", str(s))
    return ast.literal_eval(cleaned)

def _parse_zone_dict(s):
    if pd.isna(s): return None
    return ast.literal_eval(str(s))


def load_athletes(path):
    df = pd.read_csv(path)
    parsed = df["hrv_range"].apply(_parse_hrv_range)
    df["hrv_min"] = parsed.apply(lambda t: t[0])
    df["hrv_max"] = parsed.apply(lambda t: t[1])
    df.drop(columns=["hrv_range"], inplace=True)
    parsed_zones = df["hr_zones"].apply(_parse_hr_zones_athlete)
    for z in range(1, 7):
        key = f"Z{z}"
        df[f"hr_zone{z}_lo"] = parsed_zones.apply(lambda d, k=key: float(d[k][0]))
        df[f"hr_zone{z}_hi"] = parsed_zones.apply(lambda d, k=key: float(d[k][1]))
    df.drop(columns=["hr_zones"], inplace=True)
    return df

def load_daily(path):
    df = pd.read_csv(path, parse_dates=["date"])
    df.sort_values(["athlete_id", "date"], inplace=True)
    return df.reset_index(drop=True)

def load_activities(path):
    df = pd.read_csv(path, parse_dates=["date"])
    df.sort_values(["athlete_id", "date"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    parsed_hr = df["hr_zones"].apply(_parse_zone_dict)
    for z in range(1, 7):
        key = f"Z{z}"
        df[f"hr_z{z}_pct"] = parsed_hr.apply(lambda d, k=key: float(d[k]) if d else 0.0)
    df.drop(columns=["hr_zones"], inplace=True)
    parsed_pwr = df["power_zones"].apply(_parse_zone_dict)
    for z in range(1, 8):
        key = f"Z{z}"
        df[f"pwr_z{z}_pct"] = parsed_pwr.apply(lambda d, k=key: float(d[k]) if d else 0.0)
    df.drop(columns=["power_zones"], inplace=True)
    return df

def aggregate_activities(df_act):
    zone_cols = [f"hr_z{z}_pct" for z in range(1, 7)] + [f"pwr_z{z}_pct" for z in range(1, 8)]
    sport_dummies = pd.get_dummies(df_act["sport"], prefix="n")
    df_act = pd.concat([df_act, sport_dummies], axis=1)
    sport_count_cols = [c for c in df_act.columns if c.startswith("n_")]
    def _dominant(sub): return sub.groupby("sport")["tss"].sum().idxmax()
    dominant = df_act.groupby(["athlete_id", "date"]).apply(_dominant, include_groups=False).rename("dominant_sport").reset_index()
    sports_list = df_act.groupby(["athlete_id", "date"])["sport"].apply(lambda x: ",".join(sorted(x.unique()))).rename("sports_of_day").reset_index()
    agg_dict = {"tss": "sum", "duration_minutes": "sum", "work_kilojoules": "sum",
                "intensity_factor": "mean", "avg_hr": "mean",
                "training_effect_aerobic": "mean", "training_effect_anaerobic": "mean"}
    for col in zone_cols: agg_dict[col] = "mean"
    for col in sport_count_cols: agg_dict[col] = "sum"
    agg = df_act.groupby(["athlete_id", "date"]).agg(agg_dict).reset_index()
    for sport in ["bike", "run", "swim", "strength"]:
        if f"n_{sport}" not in agg.columns: agg[f"n_{sport}"] = 0
    agg = agg.merge(dominant, on=["athlete_id", "date"], how="left")
    agg = agg.merge(sports_list, on=["athlete_id", "date"], how="left")
    return agg

def build_merged(df_daily, df_act_agg, df_athletes):
    df = df_daily.merge(df_act_agg, on=["athlete_id", "date"], how="left")
    num_cols = [c for c in df_act_agg.columns if c not in ("athlete_id", "date", "dominant_sport", "sports_of_day")]
    df[num_cols] = df[num_cols].fillna(0)
    df["dominant_sport"] = df["dominant_sport"].fillna("rest")
    df["sports_of_day"] = df["sports_of_day"].fillna("")
    df_athletes = df_athletes.rename(columns={"resting_hr": "baseline_rhr", "sleep_quality": "baseline_sleep_quality"})
    df = df.merge(df_athletes, on="athlete_id", how="left")
    return df.sort_values(["athlete_id", "date"]).reset_index(drop=True)

In [ ]:
t0 = time.time()
ath = load_athletes(f"{DATA_DIR}/athletes.csv")
print(f"athletes: {ath.shape}")
daily = load_daily(f"{DATA_DIR}/daily_data.csv")
print(f"daily: {daily.shape}")
act = load_activities(f"{DATA_DIR}/activity_data.csv")
print(f"activities: {act.shape}")

act_agg = aggregate_activities(act)
merged = build_merged(daily, act_agg, ath)
print(f"merged: {merged.shape} ({time.time()-t0:.1f}s)")

## Feature engineering

In [ ]:
MIN_HISTORY_DAYS = 28
LOAD_CLASSES = ["Undertrained", "Balanced", "Overreaching"]

FEATURE_COLS = [
    "acwr", "hrv_zscore", "sleep_composite_z", "rhr_trend",
    "body_battery_morning", "stress", "sleep_hours", "deep_sleep",
    "rem_sleep", "sleep_quality", "hrv", "resting_hr",
    "tss", "duration_minutes", "intensity_factor",
    "training_effect_aerobic", "training_effect_anaerobic",
    "age", "vo2max", "ftp", "training_experience",
    "weekly_training_hours", "gender_enc", "lifestyle_enc",
]

def _rolling_slope(series, window=7):
    slopes = np.full(len(series), np.nan)
    vals = series.values
    for i in range(len(vals)):
        start = max(0, i - window + 1)
        chunk = vals[start:i+1]
        chunk = chunk[~np.isnan(chunk)]
        if len(chunk) < 3: continue
        x = np.arange(len(chunk), dtype=float)
        xm = x - x.mean()
        slopes[i] = np.dot(xm, chunk - chunk.mean()) / np.dot(xm, xm)
    return pd.Series(slopes, index=series.index)

def _engineer_athlete(df):
    df = df.copy()
    # ACWR
    acute = df["tss"].rolling(7, min_periods=1).mean()
    chronic = df["tss"].rolling(28, min_periods=7).mean()
    df["acwr"] = (acute / chronic.replace(0, np.nan)).clip(0, 3)
    # HRV z-score
    hrv_14m = df["hrv"].rolling(14, min_periods=5).mean()
    hrv_14s = df["hrv"].rolling(14, min_periods=5).std().replace(0, np.nan)
    df["hrv_zscore"] = (df["hrv"] - hrv_14m) / hrv_14s
    # sleep composite
    raw_sleep = df["sleep_hours"] * df["sleep_quality"]
    mu, sd = raw_sleep.mean(), raw_sleep.std()
    df["sleep_composite_z"] = (raw_sleep - mu) / (sd if sd > 0 else 1)
    # RHR trend
    df["rhr_trend"] = _rolling_slope(df["resting_hr"], window=7)
    # grit score (high = overreaching/dangerous, low = fresh)
    hrv_sub = 1 / (1 + np.exp(df["hrv_zscore"]))
    sleep_sub = 1 / (1 + np.exp(df["sleep_composite_z"]))
    bat_sub = 1 - (df["body_battery_morning"] / 100).clip(0, 1)
    stress_max = df["stress"].max() if df["stress"].max() > 0 else 1
    stress_sub = (df["stress"] / stress_max).clip(0, 1)
    acwr_sub = (df["acwr"] - 1.0).abs().clip(0, 1)
    df["grit_score"] = 100 * (0.25*hrv_sub + 0.25*sleep_sub + 0.20*bat_sub + 0.15*stress_sub + 0.15*acwr_sub)
    return df

def engineer_features(df_merged):
    gender_enc = LabelEncoder().fit(df_merged["gender"])
    lifestyle_enc = LabelEncoder().fit(df_merged["lifestyle"])
    df_merged = df_merged.copy()
    df_merged["gender_enc"] = gender_enc.transform(df_merged["gender"])
    df_merged["lifestyle_enc"] = lifestyle_enc.transform(df_merged["lifestyle"])
    parts = []
    for _, grp in df_merged.groupby("athlete_id", sort=False):
        parts.append(_engineer_athlete(grp.sort_values("date")))
    out = pd.concat(parts).sort_values(["athlete_id", "date"]).reset_index(drop=True)
    q_low, q_high = out["grit_score"].quantile(0.25), out["grit_score"].quantile(0.75)
    def _gc(v):
        if pd.isna(v): return 1
        if v >= q_high: return 2
        if v <= q_low: return 0
        return 1
    out["load_class"] = out["grit_score"].apply(_gc)
    out["grit_q_low"], out["grit_q_high"] = q_low, q_high
    return out

def get_feature_matrix(df_feat):
    df = df_feat.groupby("athlete_id", group_keys=False).apply(lambda g: g.iloc[MIN_HISTORY_DAYS:])
    needed = FEATURE_COLS + ["injury", "grit_score", "load_class", "athlete_id"]
    df = df[needed].dropna().reset_index(drop=True)
    X = df[FEATURE_COLS].values.astype(np.float32)
    y_class = df["injury"].values.astype(np.int64)
    y_grit = df["grit_score"].values.astype(np.float32)
    y_load = df["load_class"].values.astype(np.int64)
    groups = df["athlete_id"].values
    return X, y_class, y_grit, y_load, groups, FEATURE_COLS

In [ ]:
t1 = time.time()
df_feat = engineer_features(merged)

for col in ("acwr", "hrv_zscore", "sleep_composite_z", "rhr_trend", "grit_score"):
    s = df_feat[col].dropna()
    print(f"{col:<22} mean={s.mean():7.3f}  std={s.std():6.3f}")

print(f"\nInjury prevalence: {100*df_feat['injury'].mean():.2f}%")
print(f"Done ({time.time()-t1:.1f}s)")

In [ ]:
X, y_injury, y_grit, y_load, groups, feat_names = get_feature_matrix(df_feat)
print(f"X: {X.shape}, {len(np.unique(groups))} athletes")
print(f"injury: {np.bincount(y_injury)}, load: {np.bincount(y_load)}")
print(f"grit: mean={y_grit.mean():.1f} std={y_grit.std():.1f}")

## Model definitions

In [ ]:
N_FOLDS = 5
RANDOM_STATE = 42

class MLP(nn.Module):
    def __init__(self, in_features, out_features, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, hidden//2), nn.BatchNorm1d(hidden//2), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden//2, hidden//4), nn.BatchNorm1d(hidden//4), nn.ReLU(),
            nn.Linear(hidden//4, out_features),
        )
    def forward(self, x): return self.net(x)


def _train_mlp(X_train, y_train, X_val, task, n_classes=1, hidden=128, epochs=30, batch_size=1024, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train).astype(np.float32)
    X_vl = scaler.transform(X_val).astype(np.float32)
    out_dim = 1 if task in ("binary", "regression") else n_classes
    model = MLP(X_tr.shape[1], out_dim, hidden=hidden).to(device)
    if task == "binary":
        pw = torch.tensor([(y_train==0).sum() / max((y_train==1).sum(), 1)], dtype=torch.float32).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    elif task == "multiclass": criterion = nn.CrossEntropyLoss()
    else: criterion = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    Xt = torch.from_numpy(X_tr)
    yt = torch.from_numpy(y_train.astype(np.float32)).unsqueeze(1) if task in ("binary","regression") else torch.from_numpy(y_train.astype(np.int64))
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = criterion(model(xb), yb); loss.backward(); opt.step()
        sched.step()
    model.eval()
    return model, scaler, X_vl, device


def _predict_mlp(model, X_val_scaled, device, task, n_classes=1):
    with torch.no_grad():
        logits = model(torch.from_numpy(X_val_scaled).to(device)).cpu().numpy()
    if task == "binary":
        proba = 1 / (1 + np.exp(-logits.squeeze()))
        return (proba >= 0.5).astype(int), proba
    elif task == "multiclass":
        from scipy.special import softmax
        proba = softmax(logits, axis=1)
        return proba.argmax(axis=1), proba
    else:
        return logits.squeeze(), None


def _clf_metrics(y_true, y_pred, y_proba, n_classes):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_proba) if n_classes == 2 else roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")
    except: auc = np.nan
    return {"accuracy": acc, "f1_macro": f1, "precision": prec, "recall": rec, "roc_auc": auc}

def _reg_metrics(y_true, y_pred):
    return {"rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
            "mae": mean_absolute_error(y_true, y_pred),
            "r2": r2_score(y_true, y_pred)}

## Train all 9 models

In [ ]:
def run_clf_cv(X, y, groups, n_classes=2, label="injury"):
    gkf = GroupKFold(n_splits=N_FOLDS)
    task = "binary" if n_classes == 2 else "multiclass"
    results = {name: {"fold_metrics": []} for name in ("lr", "rf", "mlp")}
    rf_imps = []
    X_sc = StandardScaler().fit_transform(X)
    for fold, (tr, vl) in enumerate(gkf.split(X, y, groups)):
        X_tr, X_vl, y_tr, y_vl = X_sc[tr], X_sc[vl], y[tr], y[vl]
        print(f"  {label} fold {fold+1}/{N_FOLDS}", flush=True)
        # LR
        lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE, C=0.5)
        lr.fit(X_tr, y_tr)
        prob_lr = lr.predict_proba(X_vl) if n_classes > 2 else lr.predict_proba(X_vl)[:,1]
        results["lr"]["fold_metrics"].append(_clf_metrics(y_vl, lr.predict(X_vl), prob_lr, n_classes))
        # RF
        rf = RandomForestClassifier(n_estimators=200, max_depth=12, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
        rf.fit(X_tr, y_tr)
        prob_rf = rf.predict_proba(X_vl) if n_classes > 2 else rf.predict_proba(X_vl)[:,1]
        results["rf"]["fold_metrics"].append(_clf_metrics(y_vl, rf.predict(X_vl), prob_rf, n_classes))
        rf_imps.append(rf.feature_importances_)
        # MLP
        model, _, X_vl_sc, dev = _train_mlp(X[tr], y_tr, X[vl], task=task, n_classes=n_classes)
        pred_mlp, prob_mlp = _predict_mlp(model, X_vl_sc, dev, task, n_classes)
        results["mlp"]["fold_metrics"].append(_clf_metrics(y_vl, pred_mlp, prob_mlp, n_classes))
    for name in ("lr", "rf", "mlp"):
        folds = results[name]["fold_metrics"]
        results[name]["mean"] = {k: np.mean([f[k] for f in folds]) for k in folds[0]}
        results[name]["std"] = {k: np.std([f[k] for f in folds]) for k in folds[0]}
    results["feature_importance"] = np.mean(rf_imps, axis=0)
    return results

def run_reg_cv(X, y, groups, label="grit"):
    gkf = GroupKFold(n_splits=N_FOLDS)
    results = {name: {"fold_metrics": []} for name in ("lasso", "rf", "mlp")}
    rf_imps = []
    for fold, (tr, vl) in enumerate(gkf.split(X, y, groups)):
        X_tr, X_vl, y_tr, y_vl = X[tr], X[vl], y[tr], y[vl]
        print(f"  {label} fold {fold+1}/{N_FOLDS}", flush=True)
        # Lasso + poly
        pipe = Pipeline([("sc", StandardScaler()), ("poly", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)), ("lasso", Lasso(alpha=0.01, max_iter=5000))])
        pipe.fit(X_tr, y_tr)
        results["lasso"]["fold_metrics"].append(_reg_metrics(y_vl, pipe.predict(X_vl)))
        # RF
        rf = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1)
        rf.fit(X_tr, y_tr)
        results["rf"]["fold_metrics"].append(_reg_metrics(y_vl, rf.predict(X_vl)))
        rf_imps.append(rf.feature_importances_)
        # MLP
        model, _, X_vl_sc, dev = _train_mlp(X_tr, y_tr, X_vl, task="regression")
        pred, _ = _predict_mlp(model, X_vl_sc, dev, "regression")
        results["mlp"]["fold_metrics"].append(_reg_metrics(y_vl, pred))
    for name in ("lasso", "rf", "mlp"):
        folds = results[name]["fold_metrics"]
        results[name]["mean"] = {k: np.mean([f[k] for f in folds]) for k in folds[0]}
        results[name]["std"] = {k: np.std([f[k] for f in folds]) for k in folds[0]}
    results["feature_importance"] = np.mean(rf_imps, axis=0)
    return results

In [ ]:
%%time
print("Training all models...")
injury_clf = run_clf_cv(X, y_injury, groups, n_classes=2, label="injury")
print()
load_clf = run_clf_cv(X, y_load, groups, n_classes=3, label="load")
print()
grit_reg = run_reg_cv(X, y_grit, groups, label="grit")

raw_results = {"injury_clf": injury_clf, "load_clf": load_clf, "grit_reg": grit_reg, "feature_names": feat_names}
print("\nDone!")

In [ ]:
# print results
for task_name, task_key in [("Injury (binary)", "injury_clf"), ("Load Class (3-class)", "load_clf")]:
    print(f"\n== {task_name} ==")
    for key, name in [("lr", "LR"), ("rf", "RF"), ("mlp", "DNN")]:
        m = raw_results[task_key][key]["mean"]
        print(f"  {name:<5} AUC={m['roc_auc']:.3f}  F1={m['f1_macro']:.3f}  Acc={m['accuracy']:.3f}")

print(f"\n== Grit Score Regression ==")
for key, name in [("lasso", "Lasso"), ("rf", "RF"), ("mlp", "DNN")]:
    m = raw_results["grit_reg"][key]["mean"]
    print(f"  {name:<6} RMSE={m['rmse']:.3f}  R2={m['r2']:.3f}")

## Hyperparameter sweep

In [ ]:
%%time
# HP sweep with 3-fold CV for speed
gkf3 = GroupKFold(n_splits=3)
X_sc = StandardScaler().fit_transform(X)

def cv_clf(model_fn):
    scores = []
    for tr, vl in gkf3.split(X, y_load, groups):
        m = model_fn(); m.fit(X_sc[tr], y_load[tr])
        try: s = roc_auc_score(y_load[vl], m.predict_proba(X_sc[vl]), multi_class="ovr", average="macro")
        except: s = np.nan
        scores.append(s)
    return np.nanmean(scores), np.nanstd(scores)

def cv_reg(model_fn):
    scores = []
    for tr, vl in gkf3.split(X, y_grit, groups):
        m = model_fn(); m.fit(X[tr], y_grit[tr])
        scores.append(r2_score(y_grit[vl], m.predict(X[vl])))
    return np.nanmean(scores), np.nanstd(scores)

def cv_lasso(alpha):
    scores = []
    for tr, vl in gkf3.split(X, y_grit, groups):
        pipe = Pipeline([("sc", StandardScaler()), ("poly", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)), ("lasso", Lasso(alpha=alpha, max_iter=5000))])
        pipe.fit(X[tr], y_grit[tr])
        scores.append(r2_score(y_grit[vl], pipe.predict(X[vl])))
    return np.nanmean(scores), np.nanstd(scores)

def cv_mlp_clf(hidden):
    scores = []
    for tr, vl in gkf3.split(X, y_load, groups):
        model, _, xvs, dev = _train_mlp(X[tr], y_load[tr], X[vl], task="multiclass", n_classes=3, hidden=hidden, epochs=20)
        _, proba = _predict_mlp(model, xvs, dev, "multiclass", 3)
        try: s = roc_auc_score(y_load[vl], proba, multi_class="ovr", average="macro")
        except: s = np.nan
        scores.append(s)
    return np.nanmean(scores), np.nanstd(scores)

def cv_mlp_reg(hidden):
    scores = []
    for tr, vl in gkf3.split(X, y_grit, groups):
        model, _, xvs, dev = _train_mlp(X[tr], y_grit[tr], X[vl], task="regression", hidden=hidden, epochs=20)
        pred, _ = _predict_mlp(model, xvs, dev, "regression")
        scores.append(r2_score(y_grit[vl], pred))
    return np.nanmean(scores), np.nanstd(scores)

hp_results = {}

print("LR C...")
rows = []
for c in [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]:
    mu, sd = cv_clf(lambda c=c: LogisticRegression(C=c, max_iter=1000, class_weight="balanced", random_state=42))
    rows.append({"C": c, "mean_auc": mu, "std_auc": sd})
    print(f"  C={c:<8} AUC={mu:.4f}")
hp_results["lr_C"] = pd.DataFrame(rows)

print("RF clf depth...")
rows = []
for d in [3, 5, 8, 12, 16, 20]:
    mu, sd = cv_clf(lambda d=d: RandomForestClassifier(n_estimators=100, max_depth=d, class_weight="balanced", random_state=42, n_jobs=-1))
    rows.append({"max_depth": d, "mean_auc": mu, "std_auc": sd})
    print(f"  depth={d:<5} AUC={mu:.4f}")
hp_results["rf_clf_depth"] = pd.DataFrame(rows)

print("DNN clf hidden...")
rows = []
for h in [32, 64, 128, 256]:
    mu, sd = cv_mlp_clf(h)
    rows.append({"hidden_size": h, "mean_auc": mu, "std_auc": sd})
    print(f"  hidden={h:<5} AUC={mu:.4f}")
hp_results["dnn_clf_hidden"] = pd.DataFrame(rows)

print("Lasso alpha...")
rows = []
for a in [0.001, 0.01, 0.1, 0.5, 1.0, 5.0]:
    mu, sd = cv_lasso(a)
    rows.append({"alpha": a, "mean_r2": mu, "std_r2": sd})
    print(f"  alpha={a:<8} R2={mu:.4f}")
hp_results["lasso_alpha"] = pd.DataFrame(rows)

print("RF reg depth...")
rows = []
for d in [3, 5, 8, 12, 16, 20]:
    mu, sd = cv_reg(lambda d=d: RandomForestRegressor(n_estimators=100, max_depth=d, random_state=42, n_jobs=-1))
    rows.append({"max_depth": d, "mean_r2": mu, "std_r2": sd})
    print(f"  depth={d:<5} R2={mu:.4f}")
hp_results["rf_reg_depth"] = pd.DataFrame(rows)

print("DNN reg hidden...")
rows = []
for h in [32, 64, 128, 256]:
    mu, sd = cv_mlp_reg(h)
    rows.append({"hidden_size": h, "mean_r2": mu, "std_r2": sd})
    print(f"  hidden={h:<5} R2={mu:.4f}")
hp_results["dnn_reg_hidden"] = pd.DataFrame(rows)

print("\nHP sweep done!")

## Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from math import pi

PLOTS_DIR = "/content/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

plt.style.use("dark_background")
BG = "#0d1117"; PBG = "#161b22"; GC = "#30363d"; TC = "#e6edf3"
BLUE = "#4A90D9"; RED = "#D94040"; YELLOW = "#F5C842"; WHITE = "#FFFFFF"
MCOL = {"LR": BLUE, "RF": RED, "DNN": YELLOW, "Lasso": WHITE}

plt.rcParams.update({"figure.facecolor": BG, "axes.facecolor": PBG, "axes.edgecolor": GC,
    "axes.labelcolor": TC, "axes.titlecolor": TC, "xtick.color": TC, "ytick.color": TC,
    "text.color": TC, "grid.color": GC, "grid.linestyle": "--", "grid.alpha": 0.5,
    "legend.facecolor": PBG, "legend.edgecolor": GC, "font.size": 11})

fi_injury = pd.Series(raw_results["injury_clf"]["feature_importance"], index=feat_names).nlargest(15).reset_index().rename(columns={"index": "Feature", 0: "Importance"})
fi_grit = pd.Series(raw_results["grit_reg"]["feature_importance"], index=feat_names).nlargest(15).reset_index().rename(columns={"index": "Feature", 0: "Importance"})
df_sample = df_feat.sample(min(5000, len(df_feat)), random_state=42)

# feature category colors
FC = {"deep_sleep": BLUE, "sleep_quality": BLUE, "sleep_hours": BLUE, "rem_sleep": BLUE,
      "sleep_composite_z": BLUE, "hrv_zscore": WHITE, "hrv": WHITE, "rhr_trend": WHITE,
      "resting_hr": WHITE, "acwr": YELLOW, "tss": YELLOW, "duration_minutes": YELLOW,
      "intensity_factor": YELLOW, "training_effect_aerobic": YELLOW,
      "training_effect_anaerobic": YELLOW, "weekly_training_hours": YELLOW,
      "stress": RED, "body_battery_morning": RED}
def fc(f): return FC.get(f, "#8b949e")

def save(fig, name):
    fig.savefig(f"{PLOTS_DIR}/{name}", dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  {name}")

In [ ]:
# plots 01-02: classification bar charts
metrics = ["roc_auc", "f1_macro", "precision", "recall"]
mlabels = ["ROC-AUC", "F1", "Precision", "Recall"]

for fname, task_key, title, ylim in [
    ("01_injury_clf_comparison.png", "injury_clf", "Injury Classification", (0.6, 1.02)),
    ("02_load_clf_comparison.png", "load_clf", "Load Classification", (0.85, 1.02)),
]:
    models = [("LR", "lr"), ("RF", "rf"), ("DNN", "mlp")]
    x = np.arange(len(metrics)); w = 0.22
    fig, ax = plt.subplots(figsize=(10, 6), facecolor=BG); ax.set_facecolor(PBG)
    for i, (mn, mk) in enumerate(models):
        means = [raw_results[task_key][mk]["mean"][m] for m in metrics]
        stds = [raw_results[task_key][mk]["std"][m] for m in metrics]
        bars = ax.bar(x + (i-1)*w, means, w*0.9, label=mn, color=MCOL[mn],
                      yerr=stds, capsize=4, error_kw={"ecolor": TC, "alpha": 0.7}, alpha=0.88, zorder=3)
        for bar, val in zip(bars, means):
            ax.text(bar.get_x()+bar.get_width()/2, val+0.005, f"{val:.3f}", ha="center", va="bottom", fontsize=8, color=TC)
    ax.set_xticks(x); ax.set_xticklabels(mlabels); ax.set_ylim(*ylim)
    ax.set_ylabel("Score"); ax.set_title(title, pad=14)
    ax.legend(loc="lower right"); ax.yaxis.grid(True, zorder=0); ax.set_axisbelow(True)
    fig.tight_layout(); save(fig, fname)

In [ ]:
# plot 03: regression comparison
minfo = [("Lasso", "lasso", WHITE), ("RF", "rf", BLUE), ("DNN", "mlp", YELLOW)]
x = np.arange(3); w = 0.25
fig, ax1 = plt.subplots(figsize=(10, 6), facecolor=BG); ax1.set_facecolor(PBG)
ax2 = ax1.twinx()
rmse = [raw_results["grit_reg"][k]["mean"]["rmse"] for _, k, _ in minfo]
mae = [raw_results["grit_reg"][k]["mean"]["mae"] for _, k, _ in minfo]
r2 = [raw_results["grit_reg"][k]["mean"]["r2"] for _, k, _ in minfo]
ax1.bar(x-w, rmse, w*0.9, color=RED, alpha=0.88, zorder=3, label="RMSE")
ax1.bar(x, mae, w*0.9, color=BLUE, alpha=0.88, zorder=3, label="MAE")
b3 = ax2.bar(x+w, r2, w*0.9, color=YELLOW, alpha=0.88, zorder=3, label="R²")
ax1.set_xticks(x); ax1.set_xticklabels([n for n,_,_ in minfo])
ax1.set_ylabel("RMSE/MAE", color=RED); ax2.set_ylabel("R²", color=YELLOW)
ax1.set_ylim(0, 2.4); ax2.set_ylim(0.96, 0.995)
ax1.set_title("Grit Score Regression", pad=14)
ax1.legend(loc="upper right"); fig.tight_layout()
save(fig, "03_grit_regression_comparison.png")

In [ ]:
# plots 04-05: feature importance
for fi_df, title, fname in [
    (fi_injury, "Feature Importance - Injury (RF)", "04_feature_importance_injury.png"),
    (fi_grit, "Feature Importance - Grit Score (RF)", "05_feature_importance_grit.png"),
]:
    top = fi_df.head(10).sort_values("Importance", ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6), facecolor=BG); ax.set_facecolor(PBG)
    ax.barh(top["Feature"], top["Importance"], color=[fc(f) for f in top["Feature"]], alpha=0.88, height=0.65, zorder=3)
    ax.set_xlabel("Importance"); ax.set_title(title, pad=14)
    ax.xaxis.grid(True, zorder=0); ax.set_axisbelow(True)
    fig.tight_layout(); save(fig, fname)

In [ ]:
# plot 06: radar
cats = ["AUC", "F1", "Precision", "Recall", "Accuracy"]
angles = [i/5*2*pi for i in range(5)] + [0]
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True), facecolor=BG)
ax.set_facecolor(PBG)
for mn, mk in [("LR","lr"),("RF","rf"),("DNN","mlp")]:
    d = raw_results["injury_clf"][mk]["mean"]
    vals = [d["roc_auc"], d["f1_macro"], d["precision"], d["recall"], d["accuracy"]] + [d["roc_auc"]]
    ax.plot(angles, vals, "o-", lw=2, color=MCOL[mn], label=mn)
    ax.fill(angles, vals, alpha=0.12, color=MCOL[mn])
ax.set_xticks(angles[:-1]); ax.set_xticklabels(cats)
ax.set_ylim(0.6, 1.0); ax.set_title("Injury Clf - Radar", pad=22)
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
fig.tight_layout(); save(fig, "06_model_summary_radar.png")

In [ ]:
# plot 07: grit distribution
gs = df_sample["grit_score"].dropna()
q25, q75 = gs.quantile(0.25), gs.quantile(0.75)
fig, ax = plt.subplots(figsize=(10, 6), facecolor=BG); ax.set_facecolor(PBG)
ax.axvspan(gs.min()-2, q25, alpha=0.12, color=BLUE)
ax.axvspan(q25, q75, alpha=0.12, color=YELLOW)
ax.axvspan(q75, gs.max()+2, alpha=0.12, color=RED)
sns.histplot(gs, bins=50, kde=True, color=WHITE, alpha=0.65, ax=ax, zorder=3)
ax.axvline(q25, color=BLUE, lw=2, ls="--"); ax.axvline(q75, color=RED, lw=2, ls="--")
ax.set_xlabel("Grit Score"); ax.set_title("Grit Score Distribution", pad=14)
fig.tight_layout(); save(fig, "07_grit_score_distribution.png")

# plot 08: ACWR distribution
acwr = df_sample["acwr"].dropna()
fig, ax = plt.subplots(figsize=(10, 6), facecolor=BG); ax.set_facecolor(PBG)
ax.axvspan(acwr.min()-0.05, 0.8, alpha=0.14, color=RED)
ax.axvspan(0.8, 1.3, alpha=0.10, color=YELLOW)
ax.axvspan(1.3, acwr.max()+0.05, alpha=0.14, color=BLUE)
sns.histplot(acwr, bins=50, kde=True, color=BLUE, alpha=0.70, ax=ax, zorder=3)
ax.axvline(0.8, color=RED, lw=2, ls="--"); ax.axvline(1.3, color=BLUE, lw=2, ls="--")
ax.set_xlabel("ACWR"); ax.set_title("ACWR Distribution", pad=14)
fig.tight_layout(); save(fig, "08_acwr_distribution.png")

In [ ]:
# plot 09: violin plots
feats = ["deep_sleep", "sleep_quality", "rhr_trend", "hrv_zscore"]
flabels = ["Deep Sleep", "Sleep Quality", "RHR Trend", "HRV Z-score"]
df_plot = df_sample[feats + ["injury"]].dropna()
df_plot["Injury"] = df_plot["injury"].map({0: "No", 1: "Yes"})
fig, axes = plt.subplots(1, 4, figsize=(16, 7), facecolor=BG)
for ax, f, fl in zip(axes, feats, flabels):
    ax.set_facecolor(PBG)
    sns.violinplot(data=df_plot, x="Injury", y=f, palette={"No": BLUE, "Yes": RED}, inner="box", ax=ax, cut=0)
    ax.set_title(fl); ax.set_xlabel(""); ax.set_ylabel(fl)
fig.suptitle("Top Injury Predictors by Injury Status", fontsize=16, y=1.02)
fig.tight_layout(); save(fig, "09_injury_vs_features_violin.png")

In [ ]:
# plot 10: correlation heatmap
available = [c for c in FEATURE_COLS if c in df_sample.columns]
corr = df_sample[available].corr()
fig, ax = plt.subplots(figsize=(14, 12), facecolor=BG); ax.set_facecolor(PBG)
sns.heatmap(corr, ax=ax, cmap=sns.diverging_palette(220, 10, as_cmap=True),
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.3, linecolor=BG,
            annot=True, fmt=".2f", annot_kws={"size": 7}, cbar_kws={"shrink": 0.8})
ax.set_title("Feature Correlations", pad=14)
fig.tight_layout(); save(fig, "10_correlation_heatmap.png")

print("\nAll 10 plots done!")

In [ ]:
# HP sweep plots

# plot 11: classification HP sweep
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG)
for ax, key, hp, title, color in [
    (axes[0], "lr_C", "C", "LR: C", BLUE),
    (axes[1], "rf_clf_depth", "max_depth", "RF: depth", RED),
    (axes[2], "dnn_clf_hidden", "hidden_size", "DNN: hidden", YELLOW),
]:
    ax.set_facecolor(PBG); df = hp_results[key]
    ax.errorbar(df[hp], df["mean_auc"], yerr=df["std_auc"], marker="o", color=color, capsize=4, lw=2)
    if hp == "C": ax.set_xscale("log")
    ax.set_xlabel(hp); ax.set_ylabel("ROC-AUC"); ax.set_title(title); ax.grid(True, alpha=0.3)
    best = df["mean_auc"].idxmax()
    ax.scatter([df.loc[best, hp]], [df.loc[best, "mean_auc"]], s=200, color=YELLOW, zorder=5, marker="*")
fig.suptitle("HP Sweep - Classification", fontsize=16, y=1.03)
fig.tight_layout(); save(fig, "11_hp_sweep_classification.png")

# plot 12: regression HP sweep
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG)
for ax, key, hp, title, color in [
    (axes[0], "lasso_alpha", "alpha", "Lasso: alpha", WHITE),
    (axes[1], "rf_reg_depth", "max_depth", "RF: depth", RED),
    (axes[2], "dnn_reg_hidden", "hidden_size", "DNN: hidden", YELLOW),
]:
    ax.set_facecolor(PBG); df = hp_results[key]
    ax.errorbar(df[hp], df["mean_r2"], yerr=df["std_r2"], marker="o", color=color, capsize=4, lw=2)
    if hp == "alpha": ax.set_xscale("log")
    ax.set_xlabel(hp); ax.set_ylabel("R²"); ax.set_title(title); ax.grid(True, alpha=0.3)
    best = df["mean_r2"].idxmax()
    ax.scatter([df.loc[best, hp]], [df.loc[best, "mean_r2"]], s=200, color=YELLOW, zorder=5, marker="*")
fig.suptitle("HP Sweep - Regression", fontsize=16, y=1.03)
fig.tight_layout(); save(fig, "12_hp_sweep_regression.png")

print("HP plots done!")

## Save and download

In [ ]:
RESULTS_DIR = "/content/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# save ML results
def make_table(res, labels):
    rows = []
    for key, name in labels:
        m, s = res[key]["mean"], res[key]["std"]
        rows.append({"Model": name, **{k: f"{m[k]:.3f} +/- {s[k]:.3f}" for k in m}})
    return pd.DataFrame(rows)

clf_labels = [("lr", "LR"), ("rf", "RF"), ("mlp", "DNN")]
reg_labels = [("lasso", "Lasso"), ("rf", "RF"), ("mlp", "DNN")]

payload = {
    "raw": raw_results,
    "tables": {"clf_injury": make_table(injury_clf, clf_labels), "clf_load": make_table(load_clf, clf_labels),
               "reg_grit": make_table(grit_reg, reg_labels), "fi_injury": fi_injury, "fi_grit": fi_grit},
    "feature_names": feat_names,
    "df_feat_sample": df_sample,
}
with open(f"{RESULTS_DIR}/ml_results.pkl", "wb") as f: pickle.dump(payload, f)
with open(f"{RESULTS_DIR}/hp_sweep.pkl", "wb") as f: pickle.dump(hp_results, f)

!cp /content/plots/*.png /content/results/
print("Saved everything to /content/results/")
!ls -lh /content/results/

In [ ]:
!cd /content && zip -r triml_results.zip results/ plots/
from google.colab import files
files.download("/content/triml_results.zip")